# Lesson 2: RAG Triad of metrics

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import utils

import os
import openai
openai.api_key = utils.get_openai_api_key()

In [3]:
from trulens_eval import Tru

tru = Tru()
tru.reset_database()

🦑 Tru initialized with db url sqlite:///default.sqlite .
🛑 Secret keys may be written to the database. See the `database_redact_keys` option of `Tru` to prevent this.


In [4]:
from llama_index import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_files=["./eBook-How-to-Build-a-Career-in-AI.pdf"]
).load_data()

In [5]:
from llama_index import Document

document = Document(text="\n\n".\
                    join([doc.text for doc in documents]))

In [6]:
from utils import build_sentence_window_index

from llama_index.llms import OpenAI

llm = OpenAI(model="gpt-3.5-turbo", temperature=0.1)

sentence_index = build_sentence_window_index(
    document,
    llm,
    embed_model="local:BAAI/bge-small-en-v1.5",
    save_dir="sentence_index"
)

[nltk_data] Downloading package punkt to /tmp/llama_index...
[nltk_data]   Unzipping tokenizers/punkt.zip.


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [7]:
from utils import get_sentence_window_query_engine

sentence_window_engine = \
get_sentence_window_query_engine(sentence_index)

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [8]:
output = sentence_window_engine.query(
    "How do you create your AI portfolio?")
output.response

'Create your AI portfolio by building a collection of projects that clearly demonstrates your progression in skills over time.'

## Feedback functions

In [9]:
import nest_asyncio

nest_asyncio.apply()

In [10]:
from trulens_eval import OpenAI as fOpenAI

provider = fOpenAI()

### 1. Answer Relevance

In [11]:
from trulens_eval import Feedback

f_qa_relevance = Feedback(
    provider.relevance_with_cot_reasons,
    name="Answer Relevance"
).on_input_output()

✅ In Answer Relevance, input prompt will be set to __record__.main_input or `Select.RecordInput` .
✅ In Answer Relevance, input response will be set to __record__.main_output or `Select.RecordOutput` .


### 2. Context Relevance

In [12]:
from trulens_eval import TruLlama

context_selection = TruLlama.select_source_nodes().node.text

In [13]:
import numpy as np

f_qs_relevance = (
    Feedback(provider.qs_relevance,
             name="Context Relevance")
    .on_input()
    .on(context_selection)
    .aggregate(np.mean)
)

✅ In Context Relevance, input question will be set to __record__.main_input or `Select.RecordInput` .
✅ In Context Relevance, input statement will be set to __record__.app.query.rets.source_nodes[:].node.text .


In [14]:
import numpy as np

f_qs_relevance = (
    Feedback(provider.qs_relevance_with_cot_reasons,
             name="Context Relevance")
    .on_input()
    .on(context_selection)
    .aggregate(np.mean)
)

✅ In Context Relevance, input question will be set to __record__.main_input or `Select.RecordInput` .
✅ In Context Relevance, input statement will be set to __record__.app.query.rets.source_nodes[:].node.text .


### 3. Groundedness

In [15]:
from trulens_eval.feedback import Groundedness

grounded = Groundedness(groundedness_provider=provider)

In [16]:
f_groundedness = (
    Feedback(grounded.groundedness_measure_with_cot_reasons,
             name="Groundedness"
            )
    .on(context_selection)
    .on_output()
    .aggregate(grounded.grounded_statements_aggregator)
)

✅ In Groundedness, input source will be set to __record__.app.query.rets.source_nodes[:].node.text .
✅ In Groundedness, input statement will be set to __record__.main_output or `Select.RecordOutput` .


## Evaluation of the RAG application

In [17]:
from trulens_eval import TruLlama
from trulens_eval import FeedbackMode

tru_recorder = TruLlama(
    sentence_window_engine,
    app_id="App_1",
    feedbacks=[
        f_qa_relevance,
        f_qs_relevance,
        f_groundedness
    ]
)

In [18]:
eval_questions = []
with open('eval_questions.txt', 'r') as file:
    for line in file:
        # Remove newline character and convert to integer
        item = line.strip()
        eval_questions.append(item)

In [19]:
eval_questions

['What are the keys to building a career in AI?',
 'How can teamwork contribute to success in AI?',
 'What is the importance of networking in AI?',
 'What are some good habits to develop for a successful career?',
 'How can altruism be beneficial in building a career?',
 'What is imposter syndrome and how does it relate to AI?',
 'Who are some accomplished individuals who have experienced imposter syndrome?',
 'What is the first step to becoming good at AI?',
 'What are some common challenges in AI?',
 'Is it normal to find parts of AI challenging?']

In [20]:
eval_questions.append("How can I be successful in AI?")

In [21]:
eval_questions

['What are the keys to building a career in AI?',
 'How can teamwork contribute to success in AI?',
 'What is the importance of networking in AI?',
 'What are some good habits to develop for a successful career?',
 'How can altruism be beneficial in building a career?',
 'What is imposter syndrome and how does it relate to AI?',
 'Who are some accomplished individuals who have experienced imposter syndrome?',
 'What is the first step to becoming good at AI?',
 'What are some common challenges in AI?',
 'Is it normal to find parts of AI challenging?',
 'How can I be successful in AI?']

In [22]:
for question in eval_questions:
    with tru_recorder as recording:
        sentence_window_engine.query(question)

In [23]:
records, feedback = tru.get_records_and_feedback(app_ids=[])
records.head()

,app_id,app_json,type,record_id,input,output,tags,record_json,cost_json,perf_json,ts,Answer Relevance,Context Relevance,Groundedness,Answer Relevance_calls,Context Relevance_calls,Groundedness_calls,latency,total_tokens,total_cost
0,App_1,"{""app_id"": ""App_1"", ""tags"": ""-"", ""metadata"": {...",RetrieverQueryEngine(llama_index.query_engine....,record_hash_61860332db9eec7cb16bd6f2963f95e3,"""What are the keys to building a career in AI?""","""The keys to building a career in AI are:\n\n1...",-,"{""record_id"": ""record_hash_61860332db9eec7cb16...","{""n_requests"": 1, ""n_successful_requests"": 1, ...","{""start_time"": ""2026-11-04T22:29:01.468248"", ""...",2026-11-04T22:29:05.527915,1.0,0.8,0.960000,[{'args': {'prompt': 'What are the keys to bui...,[{'args': {'question': 'What are the keys to b...,[{'args': {'source': 'Chapter 7: A Simple Fram...,4,604,0.000968
1,App_1,"{""app_id"": ""App_1"", ""tags"": ""-"", ""metadata"": {...",RetrieverQueryEngine(llama_index.query_engine....,record_hash_895a1a485405957743c0bb6205531329,"""How can teamwork contribute to success in AI?""","""Teamwork contributes to success in AI by:\n\n...",-,"{""record_id"": ""record_hash_895a1a485405957743c...","{""n_requests"": 1, ""n_successful_requests"": 1, ...","{""start_time"": ""2026-11-04T22:29:05.659902"", ""...",2026-11-04T22:29:09.005114,1.0,0.8,0.916667,[{'args': {'prompt': 'How can teamwork contrib...,[{'args': {'question': 'How can teamwork contr...,[{'args': {'source': 'To get a project starte...,3,679,0.001079
2,App_1,"{""app_id"": ""App_1"", ""tags"": ""-"", ""metadata"": {...",RetrieverQueryEngine(llama_index.query_engine....,record_hash_f7c2789402839361cc0a90c231747561,"""What is the importance of networking in AI?""","""Networking is important in AI because the fie...",-,"{""record_id"": ""record_hash_f7c2789402839361cc0...","{""n_requests"": 1, ""n_successful_requests"": 1, ...","{""start_time"": ""2026-11-04T22:29:09.121941"", ""...",2026-11-04T22:29:12.277433,0.7,NaN,0.600000,[{'args': {'prompt': 'What is the importance o...,NaN,[{'args': {'source': 'What is the hiring proce...,3,568,0.000922
3,App_1,"{""app_id"": ""App_1"", ""tags"": ""-"", ""metadata"": {...",RetrieverQueryEngine(llama_index.query_engine....,record_hash_db4cc8fabd9033535447351bc5454208,"""What are some good habits to develop for a su...","""Good habits for a successful career include:\...",-,"{""record_id"": ""record_hash_db4cc8fabd903353544...","{""n_requests"": 1, ""n_successful_requests"": 1, ...","{""start_time"": ""2026-11-04T22:29:12.545942"", ""...",2026-11-04T22:29:14.917003,0.9,NaN,NaN,[{'args': {'prompt': 'What are some good habit...,NaN,NaN,2,483,0.000769
4,App_1,"{""app_id"": ""App_1"", ""tags"": ""-"", ""metadata"": {...",RetrieverQueryEngine(llama_index.query_engine....,record_hash_e8c5bc6491d0885153f20bcb6d9ea858,"""How can altruism be beneficial in building a ...","""Altruism can benefit your career by encouragi...",-,"{""record_id"": ""record_hash_e8c5bc6491d0885153f...","{""n_requests"": 1, ""n_successful_requests"": 1, ...","{""start_time"": ""2026-11-04T22:29:15.131001"", ""...",2026-11-04T22:29:17.013681,0.9,NaN,NaN,[{'args': {'prompt': 'How can altruism be bene...,NaN,NaN,1,509,0.000786


In [24]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
records[["input", "output"] + feedback]

,input,output,Answer Relevance,Groundedness,Context Relevance
0,"""What are the keys to building a career in AI?""","""The keys to building a career in AI are:\n\n1. **Learn foundational technical skills**\n2. **Work on meaningful AI projects**\n3. **Find the right AI job**\n4. **Build a supportive community** of friends and allies who can help you\u2014and whom you help in return.""",1.0,0.960000,0.8
1,"""How can teamwork contribute to success in AI?""","""Teamwork contributes to success in AI by:\n\n- Bringing colleagues, mentors, and managers onboard through clear communication of a project\u2019s value.\n- Helping identify flaws in ideas and improving the project before and during development.\n- Enabling people to lead larger projects, even without formal management positions.\n- Combining different strengths, including technical insight about architecture, data collection, and other key decisions.\n- Creating a productive environment, since colleagues who work hard, keep learning, and aim to benefit people can positively influence one another.\n- Supporting career growth by helping individuals develop leadership skills and take on larger responsibilities.""",1.0,0.916667,0.8
2,"""What is the importance of networking in AI?""","""Networking is important in AI because the field is rapidly evolving and job titles and responsibilities can vary widely between companies. Connecting with people already working in AI\u2014through informational interviews, existing networks, schools, or meetups\u2014helps you understand different roles, learn about companies, and prepare for a job search. It can also provide guidance from experienced professionals and create opportunities for mentorship and support.""",0.7,0.600000,NaN
3,"""What are some good habits to develop for a successful career?""","""Good habits for a successful career include:\n\n- Eating well \n- Exercising regularly \n- Getting enough sleep \n- Maintaining personal relationships \n- Working consistently \n- Continuing to learn \n- Practicing self-care \n- Helping and supporting others along the way""",0.9,NaN,NaN
4,"""How can altruism be beneficial in building a career?""","""Altruism can benefit your career by encouraging you to lift others up while pursuing your own goals. Helping people along your journey can lead to better outcomes for you as well as support a healthy, successful career.""",0.9,NaN,NaN
5,"""What is imposter syndrome and how does it relate to AI?""","""Imposter syndrome is when someone\u2014regardless of their success\u2014feels like a fraud and doubts whether they truly belong. In AI, newcomers may experience it as they enter the field, but it should not discourage them from learning and growing in the AI community.""",1.0,NaN,NaN
6,"""Who are some accomplished individuals who have experienced imposter syndrome?""","""Some accomplished individuals who have experienced imposter syndrome include:\n\n- Sheryl Sandberg, former Facebook COO \n- Michelle Obama, former U.S. first lady \n- Tom Hanks, actor \n- Mike Cannon-Brookes, Atlassian co-CEO""",NaN,NaN,NaN
7,"""What is the first step to becoming good at AI?""","""The first step is to learn foundational technical skills in AI.""",NaN,NaN,NaN
8,"""What are some common challenges in AI?""","""Common challenges in AI include:\n\n- Keeping up with rapidly changing technologies.\n- Finding suitable projects and estimating their timelines and return on investment.\n- Managing iterative projects when the time needed to reach target accuracy is uncertain.\n- Addressing performance drift after deployment.\n- Collaborating with stakeholders who lack AI expertise.\n- Determining which AI skills companies need and how to hire for them.\n- Understanding challenging research papers and avoiding technical mistakes.""",NaN,NaN,NaN
9,"""Is it normal to find parts of AI challenging?""","""Yes. It\u2019s normal to find parts of AI challenging\u2014even experienced researchers and engineers struggle with difficult concept

In [25]:
tru.get_leaderboard(app_ids=[])

,Answer Relevance,Groundedness,Context Relevance,latency,total_cost
app_id,,,,,
App_1,0.916667,0.825556,0.8,2.272727,0.000932


In [26]:
tru.run_dashboard()

Starting dashboard ...
Config file already exists. Skipping writing process.
Credentials file already exists. Skipping writing process.


Accordion(children=(VBox(children=(VBox(children=(Label(value='STDOUT'), Output())), VBox(children=(Label(valu…

Dashboard started at https://s172-29-16-104p38560.lab-aws-production.deeplearning.ai/ .


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>